In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os

import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)



In [3]:
from building.scaling import (
    ScalingRunConfig,
    load_full_arrays,
    run_experiments,
    summarize_results,
    plot_summary,
)

2026-04-13 10:39:34.782516: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-13 10:39:34.796662: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-13 10:39:34.796690: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
COLLECTION = "diff_genus"
N_SAMPLES = 20
EPOCHS = 30
PATIENCE = 7
BATCH_SIZE = 32
SEED = 42
THRESHOLD = 0.5
BUILD_MODEL = "cnn2d"
INPUT_REPR = "mel"
MODELS_DIR = ROOT / "models" / COLLECTION / ("scaling_"+INPUT_REPR)
RESULTS_FILE = MODELS_DIR / "results.jsonl"

In [5]:
config = ScalingRunConfig(
    collection=COLLECTION,
    build_model=BUILD_MODEL,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    threshold=THRESHOLD,
    models_dir=MODELS_DIR,
    results_file=RESULTS_FILE,
    input_repr=INPUT_REPR,
)

arrays = load_full_arrays(
    collection=COLLECTION,
    batch_size=BATCH_SIZE,
    seed=SEED,
    input_repr=INPUT_REPR,
)

print(f"Loaded {len(arrays.class_names)} classes:")
print(arrays.class_names)

Found 24500 files belonging to 10 classes.
Found 5250 files belonging to 10 classes.
Found 5250 files belonging to 10 classes.
Loaded 10 classes:
['Carduelis_carduelis', 'Carpodacus_erythrinus', 'Chloris_chloris', 'Fringilla_coelebs', 'Linaria_cannabina', 'Loxia_pytyopsittacus', 'Pyrrhula_pyrrhula', 'Serinus_serinus', 'Spinus_spinus', 'non_target']


In [6]:
baseline_rows = run_experiments(arrays, config, run_baseline=True, run_scaling=False)
print(f"New baseline runs: {len(baseline_rows)}")
if baseline_rows:
    print("Last baseline row:")
    print(baseline_rows[-1])

[baseline] target=Carduelis_carduelis
[baseline] n_each=2450 n_classes=2
Epoch 1/30


I0000 00:00:1776069616.206046   30926 service.cc:145] XLA service 0x77da0800bb60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776069616.206184   30926 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
I0000 00:00:1776069617.963572   30926 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1776069619.788681   30922 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion_1', 68 bytes spill stores, 68 bytes spill loads



154/154 - 5s - 35ms/step - accuracy: 0.7078 - loss: 0.6196 - precision: 0.6917 - recall: 0.6776 - val_accuracy: 0.7848 - val_loss: 0.5497 - val_precision: 0.7797 - val_recall: 0.7990
Epoch 2/30
154/154 - 1s - 3ms/step - accuracy: 0.8129 - loss: 0.5444 - precision: 0.7977 - recall: 0.7976 - val_accuracy: 0.8371 - val_loss: 0.5056 - val_precision: 0.8313 - val_recall: 0.8352
Epoch 3/30
154/154 - 0s - 3ms/step - accuracy: 0.8461 - loss: 0.4917 - precision: 0.8336 - recall: 0.8322 - val_accuracy: 0.8562 - val_loss: 0.4682 - val_precision: 0.8574 - val_recall: 0.8705
Epoch 4/30
154/154 - 1s - 4ms/step - accuracy: 0.8763 - loss: 0.4397 - precision: 0.8652 - recall: 0.8708 - val_accuracy: 0.8590 - val_loss: 0.4562 - val_precision: 0.8580 - val_recall: 0.8514
Epoch 5/30
154/154 - 1s - 3ms/step - accuracy: 0.8894 - loss: 0.4063 - precision: 0.8786 - recall: 0.8820 - val_accuracy: 0.8743 - val_loss: 0.4172 - val_precision: 0.8742 - val_recall: 0.8733
Epoch 6/30
154/154 - 1s - 4ms/step - accuracy

In [7]:

from building.scaling import print_baselines

print_baselines(arrays, RESULTS_FILE)

Target Class            | Precision |    Recall | Epochs | Timestamp
--------------------------------------------------------------------
'Carduelis_carduelis'   |    0.8865 |    0.8857 |     21 | 2026-04-13 08:40:32.244863+00:00
'Carpodacus_erythrinus' |    0.9339 |    0.9324 |     28 | 2026-04-13 08:40:51.587402+00:00
'Chloris_chloris'       |    0.9030 |    0.8990 |     30 | 2026-04-13 08:41:12.421604+00:00
'Fringilla_coelebs'     |    0.8918 |    0.8905 |     28 | 2026-04-13 08:41:30.913910+00:00
'Linaria_cannabina'     |    0.5000 |    0.5000 |     16 | 2026-04-13 08:41:42.827567+00:00
'Loxia_pytyopsittacus'  |    0.5000 |    0.5000 |     14 | 2026-04-13 08:41:55.407681+00:00
'Pyrrhula_pyrrhula'     |    0.9083 |    0.9076 |     21 | 2026-04-13 08:42:10.878573+00:00
'Serinus_serinus'       |    0.9353 |    0.9324 |     18 | 2026-04-13 08:42:25.125203+00:00
'Spinus_spinus'         |    0.5000 |    0.5000 |     13 | 2026-04-13 08:42:35.316119+00:00
'non_target'            |   MISSIN

In [ ]:
scaling_rows = run_experiments(
    arrays=arrays,
    config=config,
    n_samples=N_SAMPLES,
    k_values=range(2, len(arrays.class_names)),
    run_baseline=False,
    run_scaling=True,
)
print(f"New scaling runs: {len(scaling_rows)}")
if scaling_rows:
    print("Last scaling row:")
    print(scaling_rows[-1])

[scaling] k=2 sample=1/20
[scaling] n_each=2450 n_classes=3
Epoch 1/30
230/230 - 5s - 22ms/step - accuracy: 0.5913 - loss: 0.4975 - precision: 0.6452 - recall: 0.4767 - val_accuracy: 0.8673 - val_loss: 0.3024 - val_precision: 0.9188 - val_recall: 0.7543
Epoch 2/30
230/230 - 1s - 3ms/step - accuracy: 0.6736 - loss: 0.3926 - precision: 0.7978 - recall: 0.5826 - val_accuracy: 0.8870 - val_loss: 0.2413 - val_precision: 0.9152 - val_recall: 0.8432
Epoch 3/30
230/230 - 1s - 3ms/step - accuracy: 0.6907 - loss: 0.3638 - precision: 0.8411 - recall: 0.6004 - val_accuracy: 0.8889 - val_loss: 0.2298 - val_precision: 0.9222 - val_recall: 0.8502
Epoch 4/30
230/230 - 1s - 3ms/step - accuracy: 0.7001 - loss: 0.3442 - precision: 0.8741 - recall: 0.5989 - val_accuracy: 0.8971 - val_loss: 0.2246 - val_precision: 0.9354 - val_recall: 0.8362
Epoch 5/30
230/230 - 1s - 3ms/step - accuracy: 0.7133 - loss: 0.3313 - precision: 0.8857 - recall: 0.6049 - val_accuracy: 0.8883 - val_loss: 0.2167 - val_precision: 0.

I0000 00:00:1776070148.987494   30922 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion_1', 68 bytes spill stores, 68 bytes spill loads



307/307 - 5s - 16ms/step - accuracy: 0.3004 - loss: 0.6045 - precision: 0.3853 - recall: 0.1722 - val_accuracy: 0.5057 - val_loss: 0.4780 - val_precision: 0.7896 - val_recall: 0.3486
Epoch 2/30
307/307 - 1s - 3ms/step - accuracy: 0.4230 - loss: 0.5097 - precision: 0.6843 - recall: 0.2608 - val_accuracy: 0.5538 - val_loss: 0.4398 - val_precision: 0.8593 - val_recall: 0.3519
Epoch 3/30
307/307 - 1s - 3ms/step - accuracy: 0.4618 - loss: 0.4825 - precision: 0.7535 - recall: 0.2633 - val_accuracy: 0.6690 - val_loss: 0.4122 - val_precision: 0.8945 - val_recall: 0.3552
Epoch 4/30
307/307 - 1s - 3ms/step - accuracy: 0.4958 - loss: 0.4602 - precision: 0.8080 - recall: 0.2670 - val_accuracy: 0.6895 - val_loss: 0.3872 - val_precision: 0.8973 - val_recall: 0.3662
Epoch 5/30
307/307 - 1s - 3ms/step - accuracy: 0.5313 - loss: 0.4384 - precision: 0.8392 - recall: 0.2843 - val_accuracy: 0.6881 - val_loss: 0.3649 - val_precision: 0.8570 - val_recall: 0.4052
Epoch 6/30
307/307 - 1s - 3ms/step - accuracy

In [ ]:
baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f"Baseline recall: {baseline_metrics.recall:.4f}")
print(f"Baseline precision: {baseline_metrics.precision:.4f}")
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)